# 05 · Model Application — RWC 2027 Monte Carlo

**Input:** `models/best_model.pkl`, `data/gold/gold_test.parquet`  
**N simulations:** 10,000  

| Scenario | Method |
|----------|--------|
| Both teams in training set | GBT model with real 2024 form |
| At least one team unseen | Elo fallback (World Rugby Rankings) |

**RWC 2027 pools (Australia):**  
A: New Zealand, Australia, Chile, Hong Kong China  
B: South Africa, Italy, Georgia, Romania  
C: Argentina, Fiji, Spain, Canada  
D: Ireland, Scotland, Uruguay, Portugal  
E: France, Japan, USA, Samoa  
F: England, Wales, Tonga, Zimbabwe

In [27]:
# Setup
from pathlib import Path
import pandas as pd, numpy as np
import joblib
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.colors as mcolors
import warnings; warnings.filterwarnings('ignore')
import random

MODEL_DIR  = Path('../models')
GOLD_DIR   = Path('../data/gold')
REPORT_DIR = Path('../reports'); REPORT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

PRIMARY, SECONDARY, GREY = '#075020', '#8A7A2F', '#969490'
PLOT_WIDTH = 1000
PLOT_HEIGHT = 650

# load results & best model
results = joblib.load(MODEL_DIR / 'results.pkl')
best_model = joblib.load(MODEL_DIR / 'best_model.pkl')

gold_test = pd.read_parquet(GOLD_DIR / 'gold_test.parquet')
gold_test['date'] = pd.to_datetime(gold_test['date'])
known_teams = set(gold_test.team.unique())

print('✓ Loaded: results.pkl')
print(f'✓ Model loaded | known teams ({len(known_teams)}): {sorted(known_teams)}')

✓ Loaded: results.pkl
✓ Model loaded | known teams (23): ['Argentina', 'Australia', 'Chile', 'England', 'Fiji', 'France', 'Georgia', 'Hong Kong', 'Ireland', 'Italy', 'Japan', 'Kenya', 'Namibia', 'New Zealand', 'Portugal', 'Romania', 'Samoa', 'Scotland', 'South Africa', 'Tonga', 'USA', 'Uruguay', 'Wales']


In [ ]:
# Load pools + WR Rankings from sources/ CSV files
SOURCES_DIR = Path('../data/sources')
assert SOURCES_DIR.exists(), 'Missing data/sources/ — copy sources/ folder first'

# World Rugby Rankings Oct 2024 snapshot
df_rank = pd.read_csv(SOURCES_DIR / 'wr_rankings_oct2024_snapshot.csv')
RANKINGS = dict(zip(df_rank.team, df_rank.rating_pts))
print(f'WR Rankings loaded: {len(RANKINGS)} teams')
print(f'  Source: {df_rank.source.iloc[0]}')
print(f'  Snapshot date: {df_rank.snapshot_date.iloc[0]}')

# RWC 2027 pools
df_pools = pd.read_csv(SOURCES_DIR / 'rwc2027_pools.csv')
POOLS = {pool: list(grp.team) for pool, grp in df_pools.groupby('pool')}
print(f'\nRWC 2027 pools loaded: {len(POOLS)} pools')
for pool, teams in sorted(POOLS.items()):
    print(f'  Pool {pool}: {teams}')

# RWC 2027 KO bracket (informational)
df_ko = pd.read_csv(SOURCES_DIR / 'rwc2027_ko_bracket.csv')
print(f'\nKO bracket loaded: {len(df_ko)} matches')
print(f'  Source: {df_ko.source.iloc[0]}')


WR Rankings loaded: 25 teams
  Source: World Rugby (2024). Men's Rankings. https://www.world.rugby/rankings/mru
  Snapshot date: 2024-10-01

RWC 2027 pools loaded: 6 pools
  Pool A: ['New Zealand', 'Australia', 'Chile', 'Hong Kong China']
  Pool B: ['South Africa', 'Italy', 'Georgia', 'Romania']
  Pool C: ['Argentina', 'Fiji', 'Spain', 'Canada']
  Pool D: ['Ireland', 'Scotland', 'Uruguay', 'Portugal']
  Pool E: ['France', 'Japan', 'USA', 'Samoa']
  Pool F: ['England', 'Wales', 'Tonga', 'Zimbabwe']

KO bracket loaded: 15 matches
  Source: World Rugby (2025). RWC 2027 Knockouts. https://www.rugbyworldcup.com/2027/en/knockouts


In [6]:
FORM_COLS = [
    'rolling_form_3','rolling_form_5','rolling_form_10','rolling_margin_3',
    'h2h_winrate','consecutive_wins','days_since_prev',
    'experience','experience_diff','tournament_tier','pressure_index',
    'rwc_appearances','rwc_best_stage','rwc_cumul_score',
]
NEUTRAL = {
    'rolling_form_3': 0.5, 'rolling_form_5': 0.5, 'rolling_form_10': 0.5,
    'rolling_margin_3': 0.0, 'h2h_winrate': 0.5,
    'consecutive_wins': 0.0, 'days_since_prev': 30.0,
    'experience': 200.0, 'experience_diff': 0.0,
    'tournament_tier': 3.0, 'pressure_index': 4.0,
    'rwc_appearances': 5.0, 'rwc_best_stage': 3.0, 'rwc_cumul_score': 15.0,
}

avail_form = [c for c in FORM_COLS if c in gold_test.columns]

# use LAST 3 YEARS of data for form snapshot (more representative)
recent = gold_test[gold_test.date >= '2022-01-01']
if len(recent) == 0:
    recent = gold_test  # fallback

latest = (recent.sort_values('date')
                .groupby('team').last()[avail_form]
                .reset_index()
                .rename(columns={'team': 'team_name'}))

def get_form(team):
    row = latest[latest.team_name == team]
    return {c: float(row.iloc[0][c]) for c in avail_form} if len(row) else NEUTRAL.copy()

def elo_diff(t1, t2):
    return (RANKINGS.get(t1, 55) - RANKINGS.get(t2, 55)) * 16

def elo_p(t1, t2):
    return 1 / (1 + 10**(-elo_diff(t1, t2) / 400))

# teams with < 10 matches in recent period → sparse, blend heavily with Elo
recent_counts = recent.groupby('team').size()
SPARSE = set(recent_counts[recent_counts < 10].index)
print(f'Sparse teams (< 10 recent matches): {sorted(SPARSE)}')

def predict_prob(t1, t2, tournament='RWC 2027'):
    """Win probability for t1 vs t2.
    Blends GBT model with Elo rating for robustness:
    - Sparse teams (< 10 recent matches): 20% GBT + 80% Elo
    - Core teams: 50% GBT + 50% Elo
    - Unknown teams: 100% Elo
    Blending mitigates overfitting on sparse match records
    and anchors predictions to official WR ratings.
    """
    ep = elo_p(t1, t2)

    if t1 not in known_teams or t2 not in known_teams:
        return ep

    f = get_form(t1)
    X = pd.DataFrame({'home': [0], **{k: [v] for k, v in f.items()},
                      'elo_diff_pre': [elo_diff(t1, t2)],
                      'team': [t1], 'opponent': [t2], 'tournament': [tournament]})
    X = X.reindex(columns=model.named_steps['pre'].feature_names_in_, fill_value=0)
    try:
        gbt_p = float(model.predict_proba(X)[0, 1])
    except Exception:
        return ep

    if t1 in SPARSE or t2 in SPARSE:
        return 0.1 * gbt_p + 0.9 * ep   # mostly Elo for sparse teams
    return 0.3 * gbt_p + 0.7 * ep       # Elo-dominant for all teams

# sanity check
print(f'\n{"Match":<35} {"Blend":>6}  {"Elo":>6}  {"GBT":>6}')
for t1, t2 in [
    ('South Africa', 'New Zealand'),
    ('South Africa', 'Ireland'),
    ('South Africa', 'Georgia'),
    ('New Zealand',  'Ireland'),
    ('England',      'France'),
    ('Uruguay',      'Ireland'),
]:
    ep = elo_p(t1, t2)
    f  = get_form(t1)
    X  = pd.DataFrame({'home': [0], **{k: [v] for k, v in f.items()},
                       'elo_diff_pre': [elo_diff(t1, t2)],
                       'team': [t1], 'opponent': [t2], 'tournament': ['RWC 2027']})
    X  = X.reindex(columns=model.named_steps['pre'].feature_names_in_, fill_value=0)
    try:    gbt_p = float(model.predict_proba(X)[0, 1])
    except: gbt_p = ep
    blend = predict_prob(t1, t2)
    print(f'{t1+" vs "+t2:<35} {blend:>6.3f}  {ep:>6.3f}  {gbt_p:>6.3f}')


Sparse teams (< 10 recent matches): ['Chile', 'Fiji', 'Georgia', 'Hong Kong', 'Japan', 'Kenya', 'Namibia', 'Portugal', 'Romania', 'Samoa', 'Tonga', 'USA', 'Uruguay']

Match                                Blend     Elo     GBT
South Africa vs New Zealand          0.498   0.582   0.301
South Africa vs Ireland              0.546   0.610   0.398
South Africa vs Georgia              0.859   0.884   0.641
New Zealand vs Ireland               0.472   0.529   0.338
England vs France                    0.351   0.419   0.193
Uruguay vs Ireland                   0.131   0.138   0.066


In [9]:
# RWC 2027 Monte Carlo Simulation
# RWC 2027 Monte Carlo: 6 pools → top 2 per pool + best 4 3rd place → 16-team KO bracket

def sim_pool(teams):
    """Simulate pool — return sorted by points (winner first)."""
    pts = {t: 0 for t in teams}
    for i, t1 in enumerate(teams):
        for t2 in teams[i+1:]:
            if random.random() < predict_prob(t1, t2):
                pts[t1] += 4
            else:
                pts[t2] += 4
    return sorted(teams, key=lambda t: pts[t], reverse=True), pts

def sim_ko(t1, t2):
    return t1 if random.random() < predict_prob(t1, t2) else t2

def best_third(pool_thirds, eligible_pools):
    """Pick best 3rd-place team from eligible pools."""
    candidates = [pool_thirds[p] for p in eligible_pools if p in pool_thirds]
    if not candidates: return pool_thirds[list(pool_thirds.keys())[0]]
    return max(candidates, key=lambda t: RANKINGS.get(t, 55))

N      = 10_000
STAGES = ['qualify', 'r16', 'qf', 'sf', 'final', 'champion']
tally  = {t: {s: 0 for s in STAGES} for t in RANKINGS}

print(f'Running {N:,} simulations with official RWC 2027 bracket...')

for sim in range(N):
    if (sim + 1) % 2_500 == 0:
        print(f'  {sim+1:6d}/{N}')

    # ── pool stage ────────────────────────────────────────────────────────
    pool_results = {}
    pool_thirds  = {}
    for pool_name, teams in POOLS.items():
        ranked, pts = sim_pool(teams)
        pool_results[pool_name] = ranked
        pool_thirds[pool_name]  = ranked[2]  # 3rd place
        for t in ranked[:2]:
            tally[t]['qualify'] += 1

    # ── best 3rd-place teams (4 from 6) ───────────────────────────────────
    all_thirds = [(t, RANKINGS.get(t, 55)) for t in pool_thirds.values()]
    all_thirds.sort(key=lambda x: x[1], reverse=True)
    best4_thirds = [t for t, _ in all_thirds[:4]]
    for t in best4_thirds:
        tally[t]['qualify'] += 1

    # ── Round of 16 (official bracket) ────────────────────────────────────
    A1, A2 = pool_results['A'][0], pool_results['A'][1]
    B1, B2 = pool_results['B'][0], pool_results['B'][1]
    C1, C2 = pool_results['C'][0], pool_results['C'][1]
    D1, D2 = pool_results['D'][0], pool_results['D'][1]
    E1, E2 = pool_results['E'][0], pool_results['E'][1]
    F1, F2 = pool_results['F'][0], pool_results['F'][1]

    # best 3rd place teams assigned to bracket positions
    thirds_ranked = [t for t, _ in all_thirds[:4]]
    t3_1 = thirds_ranked[0] if len(thirds_ranked) > 0 else best4_thirds[0]
    t3_2 = thirds_ranked[1] if len(thirds_ranked) > 1 else best4_thirds[0]
    t3_3 = thirds_ranked[2] if len(thirds_ranked) > 2 else best4_thirds[0]
    t3_4 = thirds_ranked[3] if len(thirds_ranked) > 3 else best4_thirds[0]

    # R16 matches (official bracket)
    r16_1 = sim_ko(A1, t3_1)   # 1A v best3rd(C/E/F)
    r16_2 = sim_ko(B1, t3_2)   # 1B v best3rd(D/E/F)
    r16_3 = sim_ko(E1, D2)     # 1E v 2D
    r16_4 = sim_ko(F1, B2)     # 1F v 2B
    r16_5 = sim_ko(C1, t3_3)   # 1C v best3rd(A/E/F)
    r16_6 = sim_ko(D1, t3_4)   # 1D v best3rd(B/E/F)
    r16_7 = sim_ko(A2, E2)     # 2A v 2E
    r16_8 = sim_ko(C2, F2)     # 2C v 2F

    for t in [r16_1,r16_2,r16_3,r16_4,r16_5,r16_6,r16_7,r16_8]:
        tally[t]['r16'] += 1

    # ── Quarter-finals ────────────────────────────────────────────────────
    qf1 = sim_ko(r16_2, r16_4)
    qf2 = sim_ko(r16_1, r16_3)
    qf3 = sim_ko(r16_5, r16_6)
    qf4 = sim_ko(r16_7, r16_8)
    for t in [qf1, qf2, qf3, qf4]: tally[t]['qf'] += 1

    # ── Semi-finals ───────────────────────────────────────────────────────
    sf1 = sim_ko(qf1, qf2)
    sf2 = sim_ko(qf3, qf4)
    for t in [sf1, sf2]: tally[t]['sf'] += 1

    # ── Final ─────────────────────────────────────────────────────────────
    champion = sim_ko(sf1, sf2)
    runner   = sf2 if champion == sf1 else sf1
    tally[champion]['champion'] += 1
    tally[runner]['final']      += 1

print('Simulations complete.')


Running 10,000 simulations with official RWC 2027 bracket...
    2500/10000
    5000/10000
    7500/10000
   10000/10000
Simulations complete.


In [10]:
results_df = (pd.DataFrame([
    {'Team': t, 'WR_Rating': RANKINGS.get(t,55),
     'P_Qualify': d['qualify']/N, 'P_QF': d['qf']/N,
     'P_SF': d['sf']/N, 'P_Final': d['final']/N,
     'P_Champion': d['champion']/N}
    for t,d in tally.items()
]).sort_values('P_Champion', ascending=False).reset_index(drop=True))

results_df.to_csv(REPORT_DIR / 'rwc2027_results.csv', index=False)
print(results_df[['Team','P_Champion','P_Final','P_SF','P_QF']].head(12).to_string(index=False))

        Team  P_Champion  P_Final   P_SF   P_QF
South Africa      0.2385   0.1207 0.3592 0.6032
 New Zealand      0.1561   0.1045 0.2606 0.4089
     Ireland      0.1481   0.1224 0.2705 0.4917
      France      0.1085   0.0869 0.1954 0.3421
   Argentina      0.0817   0.0897 0.1714 0.3221
    Scotland      0.0648   0.0889 0.1537 0.3350
   Australia      0.0546   0.0843 0.1389 0.2570
     England      0.0501   0.0657 0.1158 0.2727
        Fiji      0.0497   0.0879 0.1376 0.2974
       Wales      0.0179   0.0439 0.0618 0.1550
       Italy      0.0120   0.0282 0.0402 0.1375
       Japan      0.0083   0.0258 0.0341 0.1012


In [ ]:
possible_opponents = [t for t in RANKINGS.keys() if t != 'South Africa']

sa_df = pd.DataFrame([
    {
        'Opponent': opp,
        'P_win': predict_prob('South Africa', opp),
        'Method': 'GBT'
    }
    for opp in possible_opponents
]).sort_values('P_win', ascending=False).reset_index(drop=True)

print(sa_df)

           Opponent     P_win Method
0            Canada  0.962353    GBT
1          Zimbabwe  0.962185    GBT
2   Hong Kong China  0.959375    GBT
3           Namibia  0.958403    GBT
4           Romania  0.936337    GBT
5             Spain  0.919405    GBT
6             Samoa  0.908849    GBT
7             Tonga  0.907539    GBT
8             Chile  0.907193    GBT
9               USA  0.891963    GBT
10          Uruguay  0.889273    GBT
11         Portugal  0.886078    GBT
12          Georgia  0.859257    GBT
13            Japan  0.836635    GBT
14            Wales  0.771847    GBT
15            Italy  0.757901    GBT
16             Fiji  0.746440    GBT
17        Australia  0.705244    GBT
18          England  0.670687    GBT
19         Scotland  0.626734    GBT
20        Argentina  0.625375    GBT
21           France  0.589946    GBT
22          Ireland  0.546472    GBT
23      New Zealand  0.498016    GBT


In [30]:
top16 = results_df.head(16).sort_values('P_Champion', ascending=False).reset_index(drop=True)

LIGHT = '#C8C0A0'

def tier_color_by_rank(idx):
    """Color by rank position (0-based)"""
    if idx < 5:      return PRIMARY     
    if idx < 10:     return SECONDARY   
    return LIGHT

colors = [tier_color_by_rank(i) for i in range(len(top16))]

fig = go.Figure()

for tier_idx, (tier_name, tier_color, tier_range) in enumerate([
    ('Top',    PRIMARY,   (0, 5)),
    ('Mid',     SECONDARY, (5, 10)),
    ('Lower',    LIGHT,     (10, 16)),
]):
    mask = (top16.index >= tier_range[0]) & (top16.index < tier_range[1])
    tier_data = top16[mask]
    
    fig.add_trace(go.Bar(
        y=tier_data.Team,
        x=tier_data.P_Champion,
        orientation='h',
        name=tier_name,
        marker=dict(color=tier_color, line=dict(width=0)),
        text=[f'{p:.1%}' for p in tier_data.P_Champion],
        textposition='outside',
        hovertemplate='%{y}<br>P(Champion): %{x:.1%}<extra></extra>',
    ))

fig.update_layout(
    xaxis_title='Championship Probability',
    xaxis=dict(tickformat='.0%', range=[0, top16.P_Champion.max() * 1.22]),
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=PLOT_HEIGHT,
    width=PLOT_WIDTH,
    margin=dict(l=60, r=60, t=60, b=60),
    hovermode='closest',
    legend=dict(x=1.1, y=1.1, xanchor='right', yanchor='top')
)

fig.show()

In [29]:
tier1 = ['South Africa','New Zealand','Ireland','France','England',
         'Australia','Scotland','Argentina','Italy','Wales']
t1 = (results_df[results_df.Team.isin(tier1)]
      .sort_values('P_Champion', ascending=False).reset_index(drop=True))

t33_t1 = t1.P_Champion.quantile(0.33)
t66_t1 = t1.P_Champion.quantile(0.66)

fig = go.Figure()

for tier_idx, (tier_name, tier_color, threshold_min, threshold_max) in enumerate([
    ('Top',   PRIMARY,   t66_t1, 1.0),
    ('Mid',   SECONDARY, t33_t1, t66_t1),
    ('Lower', LIGHT,     0.0,    t33_t1),
]):
    mask = (t1.P_Champion >= threshold_min) & (t1.P_Champion < threshold_max)
    tier_data = t1[mask]
    
    fig.add_trace(go.Bar(
        y=tier_data.Team,
        x=tier_data.P_Champion,
        orientation='h',
        name=tier_name,
        marker=dict(color=tier_color, line=dict(width=0)),
        text=[f'{p:.1%}' for p in tier_data.P_Champion],
        textposition='outside',
        hovertemplate='%{y}<br>P(Champion): %{x:.1%}<extra></extra>',
    ))

fig.update_layout(
    xaxis_title='Championship Probability',
    xaxis=dict(tickformat='.0%', range=[0, t1.P_Champion.max() * 1.22]),
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=PLOT_HEIGHT,
    width=PLOT_WIDTH,
    margin=dict(l=60, r=60, t=60, b=60),
    hovermode='closest',
    legend=dict(x=1.1, y=1.1, xanchor='right', yanchor='top')
)

fig.show()

In [38]:
from matplotlib.colors import Normalize, LinearSegmentedColormap

# Gradient colormap
bok_cmap = LinearSegmentedColormap.from_list('bok', [SECONDARY, PRIMARY])
norm = Normalize(vmin=sa_df.P_win.min(), vmax=sa_df.P_win.max())

# Map probabilities to hex colors
colors = [mcolors.to_hex(bok_cmap(norm(p))) for p in sa_df.P_win]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=sa_df.Opponent,
    x=sa_df.P_win,
    orientation='h',
    marker=dict(color=colors, line=dict(width=0)),
    text=[f'{p:.1%}  [{m}]' for p, m in zip(sa_df.P_win, sa_df.Method)],
    textposition='outside',
    hovertemplate='%{y}<br>P(win): %{x:.1%}',
    showlegend=False
))

# 50% threshold line
fig.add_vline(x=0.5, line_dash='dash', line_color='#960d0d', line_width=1, opacity=0.7)

# Annotation
fig.add_annotation(
    x=0.5, y=0.98,
    xref='x', yref='paper',
    text='50% threshold',
    showarrow=False,
    bgcolor='darkred',
    bordercolor='#960d0d',
    borderwidth=1,
    font=dict(color='white', size=8),
    align='center',
    xanchor='center',
    yanchor='top'
)

fig.update_layout(
    xaxis_title='P(win | South Africa)',
    xaxis=dict(tickformat='.0%', range=[0, 1.32]),
    template='plotly_white',
    height=PLOT_HEIGHT,
    width=PLOT_WIDTH,
    margin=dict(l=60, r=60, t=60, b=60),
    hovermode='closest'
)

fig.show()

In [34]:
sa     = results_df[results_df.Team == 'South Africa'].iloc[0]
stages = ['P_Qualify', 'P_QF', 'P_SF', 'P_Final', 'P_Champion']
labels = ['Qualify', 'Quarter-final', 'Semi-final', 'Final', 'Champion']
vals   = [sa[s] for s in stages]

# Gradient colors (normalized by max value)
vals_normalized = np.array(vals) / max(vals)
bok_cmap = LinearSegmentedColormap.from_list('bok', [SECONDARY, PRIMARY])
colors = [mcolors.to_hex(bok_cmap(v)) for v in vals_normalized]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=labels,
    y=vals,
    marker=dict(color=colors, line=dict(width=0)),
    text=[f'{v:.1%}' for v in vals],
    textposition='outside',
    hovertemplate='Probability: %{y:.1%}<extra></extra>',
    showlegend=False
))

fig.update_layout(
    yaxis_title='Probability',
    yaxis=dict(tickformat='.0%', range=[0, 1.12]),
    xaxis=dict(tickangle=0),
    template='plotly_white',
    height=PLOT_HEIGHT,
    width=PLOT_WIDTH,
    margin=dict(l=60, r=60, t=60, b=60),
    hovermode='x unified'
)

fig.show()